# Lab 04: Getting Started with Remote Repos

> Tuesday, Sept 22nd
>
> GSIs: Sequoia Andrade & Larissa Arreola

---

## Before you start

This lab assumes:
- You already have a GitHub account
- You're comfortable with local git: `init`, `add`, `commit`, `branch`
- You've authenticated GitHub from this environment before

(We can help out if something out of those 3 points isn't clear)


## Part 1: Your Own Remote Repo

### 1.1 Two ways to connect a repo to GitHub

There are two situations you'll run into constantly: starting a brand-new project (on GitHub), and connecting a project you already have locally. Let's do both — you'll end this section with two small repos of your own.

### Path A — Start on GitHub, then clone

1. On GitHub, create a new repository under your own account (add a README and the .gitignore file by choosing the language you are using for your repo, this doesn't really matter in the long run, because you can have multiple types of scripts, but it's a starting point).
2. Copy the repo's URL (the "Code" button).
3. In your terminal:
   ```bash
   git clone <repo_url>
   ```
4. `cd` into the new folder, create a text file with a line or two in it, then add, commit, and push:
   ```bash
   git add <file>
   git commit -m "add a first file"
   git push
   ```
5. On GitHub (the website), refresh and confirm your file is there.

### Path B — Start locally, then connect to GitHub

1. Create a new folder, `cd` into it, and run `git init`.
2. Add a text file, then `git add` and `git commit` it.
3. On GitHub, create a new **empty** repo (except this time, opposite to Path A, we want the repository empty, so don't add a README nor a .gitignore).
4. Connect the two:
   ```bash
   git remote add origin <repo_url>
   git push -u origin main
   ```
5. Refresh the GitHub page and confirm your commit is there.

> **Notice the `-u` in Path B's push?** That's short for `--set-upstream` — it tells git "remember that my local `main` branch corresponds to `origin`'s `main` branch." After this one push, plain `git push` and `git pull` both work with no extra flags. We'll come back to this in 1.4.

Choose any of the two repos you just created, it doesn't matter which one.
In the following cell you have some code to generate a dataframe, copy it in a separate script and save it to a (new) "data/" directory.

In [1]:
import pandas as pd
import numpy as np

np.random.seed(42)
data = {
    'Transaction_ID': range(1001, 1101),
    'Store_ID': np.random.choice(['Store_A', 'Store_B', 'Store_C', None], 100, p=[0.4, 0.3, 0.2, 0.1]),
    'Revenue': np.random.choice([50, 120, 250, 500, np.nan], 100, p=[0.3, 0.4, 0.15, 0.10, 0.05]),
    'Units_Sold': np.random.randint(1, 12, size=100)
}
df = pd.DataFrame(data)

Commit this change (the script, the new generated data).
Let's continue.

In another script (that will live under a "scripts/" directory:
- read the raw dataframe you previously generated
- check how many missing values per column you have
- drop NAs for Store IDs (remember to not directly modify the "raw" dataframe, create a new --clean-- one to store the df after you dropped values, that's your working df now).
- *commit changes*
- For the rows with missing Revenue, impute it through the median of its specific Unit count group
- For those cases that weren't caught in the last step, impute the NAs with the overall median
- check if there are any other missing values
- save this clean df under the data/ directory
- *commit changes*


Now create a new script, under the same scripts/ directory:
- read the cleaned df
- create a new column for the unit price, that you will populate by dividing the revenue column by the units sold column
- *commit changes*
- now create a new column called "order_segment", where you label the row using the following conditions:
```
conditions = [
    (df_clean['Revenue'] >= 300) & (df_clean['Units_Sold'] >= 5),
    (df_clean['Revenue'] >= 100),
    (df_clean['Revenue'] < 100)
]
choices = ['Bulk High-Value', 'Standard Retail', 'Low-Margin']
```
- inspect your new df, save it in the data/ folder
- *commit changes*
- *push to remote*

Finally, in a new script, in the same script/ folder
- read the last df
- create a store summary (group by store id, and aggregate the total revenue, avg units sold and the unique order segments)
- *commit changes*
- Identify transactions 1.5 IQR above the median revenue (you can use the quantile function in pandas)
- print the store summary performance and the outliers detected in a text file and store it under a "results/" folder.
- *commit changes*
- *push to remote*


### *`origin`

From inside either repo you just made, run:
```bash
git remote -v
```
- You'll see `origin` listed twice — once for fetch, once for push. That's normal; they're almost always identical.
- `origin` isn't a keyword. It's just the conventional name git gives your first remote. You could name it anything.
- A repo can have more than one remote at once. You'll add a second one, `upstream`, in Part 2.

**Try it — see how the name is arbitrary:**
```bash
git remote rename origin upstream-test
git remote -v
git remote rename upstream-test origin
```


## Part 2: Push errors you might encounter

### 2.1 Manufacturing (and fixing) a rejected push

This is the error you'll hit most often in real collaboration, so let's trigger it on purpose — safely, on a repo that's entirely yours.

1. Go to one of your repos **on GitHub** and edit your file directly in the browser: click the pencil icon on the file, change a line, and commit the change **directly on GitHub**. This is the browser-editing feature from lecture — you'll come back to it more in Part 2.
2. Back in your terminal, **without pulling first**, make a small, unrelated local change. Add and commit it.
3. Try to push:
   ```bash
   git push
   ```
4. You should see something like:
   ```
   ! [rejected]        main -> main (fetch first)
   error: failed to push some refs
   ```
   Git is refusing your push because the remote has a commit — your browser edit — that your local repo doesn't know about yet.
5. Fix it the standard way:
   ```bash
   git pull
   git push
   ```
   If `git pull` opens an editor for a merge commit message, that's expected — save and close it (in most default setups: `:wq` in vim, or Ctrl+X then Y in nano).

### 2.2 "fatal: no upstream branch"

1. Create a new local branch and switch to it:
   ```bash
   git switch -c try-a-branch
   ```
2. Make a small change, then add and commit it.
3. Try to push:
   ```bash
   git push
   ```
4. You'll see:
   ```
   fatal: The current branch try-a-branch has no upstream branch.
   ```
   Git already knows about `origin` — but it doesn't know where *this particular branch* should go.
5. Fix it exactly like you did in Path B:
   ```bash
   git push -u origin try-a-branch
   ```
6. Confirm on GitHub that the new branch shows up.


### 2.3 Force-push: know it, mostly avoid it

`git push --force` overwrites the remote branch with your local history — no questions asked, no rejection, no merge. If the remote has commits you don't have locally (like in 1.3), force-pushing deletes them. Permanently — for anyone else using that branch too.

**Reflect, don't run:** in 1.3, if you'd force-pushed instead of pulling, what would have happened to your browser edit?


## Extra

- `git log --oneline --graph --all`, run inside your fork's clone. Can you spot the commit where you forked?
- Now that you have two remotes on your fork (`origin` and `upstream`), run `git remote -v` again and try to think how you would explain someone else what you are getting back.
